# Experimento 03 — Benchmark de métodos y red flags

**Objetivo.** Comparar el conjunto de anomalías detectadas por el Isolation Forest del Experimento 02 contra:

1. **Otros métodos no supervisados** (HBOS, ECOD, LOF) que aporten una segunda opinión y permitan construir un conjunto de **consenso multi-método**.
2. **Reglas heurísticas** (*red flags*) clásicas de la literatura OCDE/Banco Mundial, para evidenciar el aporte del modelo respecto a un baseline simple.

**Métricas de comparación:**
- Solapamiento Jaccard entre los conjuntos de anomalías al P5 de cada método.
- Diagrama de Venn entre IF, HBOS y ECOD (los tres que escalan al dataset completo).
- Brecha entre IF y red flags: cuántas ofertas el modelo señala que las reglas simples no.

**Métodos descartados:** LOF sobre 3M de filas (costo $O(n^2)$); se aplica solo sobre el subconjunto formado por las anomalías robustas y una muestra normal estratificada (≈158 k filas).

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from pyod.models.hbos import HBOS
    from pyod.models.ecod import ECOD
except ImportError:
    raise ImportError('Se requiere pyod. Instalar con: pip install pyod')

from sklearn.neighbors import LocalOutlierFactor

try:
    from matplotlib_venn import venn3, venn2
    HAS_VENN = True
except ImportError:
    HAS_VENN = False
    print('matplotlib_venn no disponible — los diagramas de Venn se omitirán.')
    print('Instalar con: pip install matplotlib-venn')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

DATA_PATH = Path('..') / 'datos' / 'V_BASE_OFERTA_ITEM.parquet'
EXP02_PARQUET = Path('..') / 'exp_02_refinamiento' / 'resultados' / 'anomalias_exp02.parquet'
ROBUSTAS_PATH = Path('..') / 'exp_02_refinamiento' / 'resultados' / 'anomalias_exp02_robustas.csv'

FIG_DIR = Path('figuras')
RES_DIR = Path('resultados')
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

SEMILLA_PRINCIPAL = 42
PERCENTIL = 0.05
MIN_OBS_HBOS = 50

df = pd.read_parquet(DATA_PATH)
print(f'Registros: {len(df):,}  |  Columnas: {df.shape[1]}')

Registros: 3,021,336  |  Columnas: 70


## 2. Feature engineering (idéntico a exp_02 y al notebook de SHAP)

In [2]:
df['LOG_PRECIO_OFERTADO'] = np.where(
    df['OFERTA_PRECIO_UNITARIO_CRC'] > 0,
    np.log10(df['OFERTA_PRECIO_UNITARIO_CRC']), np.nan)
df['LOG_PRECIO_ESTIMADO'] = np.where(
    df['CARTEL_PRECIO_UNITARIO_CRC'] > 0,
    np.log10(df['CARTEL_PRECIO_UNITARIO_CRC']), np.nan)
df['LOG_RATIO_VS_ESTIMADO'] = np.where(
    df['RATIO_OFERTADO_VS_ESTIMADO'] > 0,
    np.log10(df['RATIO_OFERTADO_VS_ESTIMADO']), np.nan)

for col in ['FECHA_PUBLICACION', 'FECHA_CIERRE_RECEPCION',
            'FECHA_APERTURA', 'FECHA_PRESENTA_OFERTA']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['DIAS_PARA_CIERRE'] = (df['FECHA_CIERRE_RECEPCION'] - df['FECHA_PRESENTA_OFERTA']).dt.total_seconds() / 86400
df['DIAS_VENTANA_PROCESO'] = (df['FECHA_CIERRE_RECEPCION'] - df['FECHA_PUBLICACION']).dt.total_seconds() / 86400
df['RATIO_CANTIDAD_VS_SOLICITADA'] = np.where(
    df['CARTEL_CANTIDAD_SOLICITADA'].isna() | (df['CARTEL_CANTIDAD_SOLICITADA'] == 0),
    np.nan, df['CANTIDAD_OFERTADA'] / df['CARTEL_CANTIDAD_SOLICITADA'])

oferta_prod_8 = df['OFERTA_CODIGO_PRODUCTO'].astype(str).str[:8]
cartel_prod_8 = df['CARTEL_CODIGO_PRODUCTO'].astype(str).str[:8]
df['PRODUCTO_DIFIERE'] = (oferta_prod_8 != cartel_prod_8).astype(int)

df['CV_PRECIO_ITEM'] = np.where(
    df['PRECIO_PROM_ITEM_CRC'].isna() | (df['PRECIO_PROM_ITEM_CRC'] == 0),
    np.nan, df['PRECIO_STDDEV_ITEM_CRC'] / df['PRECIO_PROM_ITEM_CRC'])

df['REGIMEN_LGCP'] = (df['FECHA_PRESENTA_OFERTA'] >= '2022-12-01').astype(int)
tamano_map = {'Microemprendedor': 1, 'Pequeña': 2, 'Mediana': 3, 'Grande': 4}
df['TAMANO_PROVEEDOR_ORD'] = df['TAMANO_PROVEEDOR'].map(tamano_map)

tp_cols = [c for c in df.columns if c.startswith('TP_')]
mp_cols = [c for c in df.columns if c.startswith('MP_')]

FEATURES_RELATIVAS = [
    'RATIO_OFERTADO_VS_ESTIMADO', 'RATIO_OFERTADO_VS_PROM_ITEM',
    'RATIO_OFERTADO_VS_MIN_ITEM', 'LOG_RATIO_VS_ESTIMADO',
    'RANK_PRECIO_ASC', 'N_OFERTAS_ITEM', 'N_OFERENTES_ITEM',
    'N_OFERENTES_ELEGIBLES_ITEM', 'SINGLE_BID_FLAG', 'CV_PRECIO_ITEM',
    'RATIO_CANTIDAD_VS_SOLICITADA', 'PRODUCTO_DIFIERE',
    'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO',
    'OFERTA_IVA', 'OFERTA_DESCUENTO', 'REGIMEN_LGCP',
] + tp_cols + mp_cols + ['TAMANO_PROVEEDOR_ORD']

binary_features = ['SINGLE_BID_FLAG', 'PRODUCTO_DIFIERE', 'REGIMEN_LGCP'] + tp_cols + mp_cols
for f in binary_features:
    if f in df.columns:
        df[f] = df[f].fillna(0)
for f in FEATURES_RELATIVAS:
    if f in df.columns and f not in binary_features and df[f].isna().any():
        df[f] = df[f].fillna(df[f].median())
if 'RANK_PRECIO_ASC' in df.columns:
    df['RANK_PRECIO_ASC'] = df['RANK_PRECIO_ASC'].astype(float)

print(f'Features para benchmark: {len(FEATURES_RELATIVAS)}')
print(f'Nulos restantes: {df[FEATURES_RELATIVAS].isna().sum().sum()}')

Features para benchmark: 18
Nulos restantes: 0


## 3. Cargar resultados del Isolation Forest (exp_02)

Se importa la columna `SCORE_MIN` y `ANOMALY_FLAG` del parquet de exp_02 al dataframe principal usando la llave `(NRO_SICOP, NRO_OFERTA, NRO_LINEA)`.

In [3]:
key_cols = ['NRO_SICOP', 'NRO_OFERTA', 'NRO_LINEA']
exp02 = pd.read_parquet(EXP02_PARQUET)
print(f'Filas en exp_02 parquet: {len(exp02):,}')
print(f'Columnas: {list(exp02.columns)[:20]}...')

score_cols_keep = [c for c in ['SCORE_MIN', 'SCORE_SEGMENTO', 'SCORE_CLASE', 'ANOMALY_FLAG']
                   if c in exp02.columns]
exp02_subset = exp02[key_cols + score_cols_keep].drop_duplicates(subset=key_cols)
df = df.merge(exp02_subset, on=key_cols, how='left', suffixes=('', '_exp02'))

if 'ANOMALY_FLAG' not in df.columns or df['ANOMALY_FLAG'].isna().all():
    umbral = df['SCORE_MIN'].quantile(PERCENTIL)
    df['ANOMALY_FLAG'] = (df['SCORE_MIN'] <= umbral).astype(int)

n_if = int(df['ANOMALY_FLAG'].sum())
print(f'\nIsolation Forest — anomalías al P{int(PERCENTIL*100)}: {n_if:,} ({n_if/len(df)*100:.2f}%)')

Filas en exp_02 parquet: 3,021,336
Columnas: ['NRO_SICOP', 'NRO_PROCEDIMIENTO', 'NRO_OFERTA', 'NRO_LINEA', 'CEDULA_INSTITUCION', 'NOMBRE_INSTITUCION', 'TIPO_PROCEDIMIENTO', 'MODALIDAD_PROCEDIMIENTO', 'CEDULA_PROVEEDOR', 'NOMBRE_PROVEEDOR', 'TAMANO_PROVEEDOR', 'OFERTA_SEGMENTO', 'OFERTA_FAMILIA', 'OFERTA_CLASE', 'OFERTA_MERCADERIA', 'OFERTA_CODIGO_PRODUCTO', 'CARTEL_CODIGO_PRODUCTO', 'OFERTA_TIPO_MONEDA', 'OFERTA_PRECIO_UNITARIO_CRC', 'CARTEL_PRECIO_UNITARIO_CRC']...

Isolation Forest — anomalías al P5: 151,067 (5.00%)


## 4. HBOS por segmento

**HBOS (Histogram-Based Outlier Score, Goldstein & Dengel 2012):** asume independencia entre features y construye un histograma por feature; el score es el producto del log-inverso de las frecuencias (alta densidad → score bajo). Es lineal en *n* y muy rápido. Se entrena por segmento UNSPSC para mantener consistencia con la arquitectura del Experimento 02.

In [4]:
df['SCORE_HBOS'] = np.nan
segmentos = sorted(df['OFERTA_SEGMENTO'].unique())

t0 = time.time()
for seg in segmentos:
    mask = (df['OFERTA_SEGMENTO'] == seg).values
    n = mask.sum()
    if n < MIN_OBS_HBOS:
        continue
    X = df.loc[mask, FEATURES_RELATIVAS].values
    model = HBOS(n_bins='auto', alpha=0.1, contamination=0.05)
    model.fit(X)
    scores = model.decision_function(X)
    df.loc[mask, 'SCORE_HBOS'] = scores

elapsed = time.time() - t0
n_with = df['SCORE_HBOS'].notna().sum()
print(f'HBOS completado en {elapsed:.1f}s')
print(f'Ofertas con SCORE_HBOS: {n_with:,} ({n_with/len(df)*100:.1f}%)')

umbral_hbos = df['SCORE_HBOS'].quantile(1 - PERCENTIL)
df['ANOM_HBOS'] = (df['SCORE_HBOS'] >= umbral_hbos).astype(int)
n_hbos = int(df['ANOM_HBOS'].sum())
print(f'HBOS — anomalías al P{int(PERCENTIL*100)}: {n_hbos:,} ({n_hbos/len(df)*100:.2f}%)')

HBOS completado en 311.6s
Ofertas con SCORE_HBOS: 3,021,336 (100.0%)
HBOS — anomalías al P5: 151,069 (5.00%)


## 5. ECOD por segmento

**ECOD (Empirical Cumulative Distribution-based Outlier Detection, Li et al. 2022):** estima la cola de la distribución empírica de cada feature por separado y agrega los scores. No tiene hiperparámetros relevantes y es paramétrico-libre. Útil como tercer punto de vista, especialmente porque el supuesto de independencia es distinto al del HBOS.

In [5]:
df['SCORE_ECOD'] = np.nan

t0 = time.time()
for seg in segmentos:
    mask = (df['OFERTA_SEGMENTO'] == seg).values
    n = mask.sum()
    if n < MIN_OBS_HBOS:
        continue
    X = df.loc[mask, FEATURES_RELATIVAS].values
    model = ECOD(contamination=0.05, n_jobs=-1)
    model.fit(X)
    scores = model.decision_function(X)
    df.loc[mask, 'SCORE_ECOD'] = scores

elapsed = time.time() - t0
n_with = df['SCORE_ECOD'].notna().sum()
print(f'ECOD completado en {elapsed:.1f}s')
print(f'Ofertas con SCORE_ECOD: {n_with:,} ({n_with/len(df)*100:.1f}%)')

umbral_ecod = df['SCORE_ECOD'].quantile(1 - PERCENTIL)
df['ANOM_ECOD'] = (df['SCORE_ECOD'] >= umbral_ecod).astype(int)
n_ecod = int(df['ANOM_ECOD'].sum())
print(f'ECOD — anomalías al P{int(PERCENTIL*100)}: {n_ecod:,} ({n_ecod/len(df)*100:.2f}%)')

[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done   2 out of  14 | elapsed:   17.1s remaining:  1.7min
[Parallel(n_jobs=14)]: Done  14 out of  14 | elapsed:   17.4s finished
[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done   2 out of  14 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=14)]: Done  14 out of  14 | elapsed:    0.0s finished
[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done   2 out of  14 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=14)]: Done  14 out of  14 | elapsed:    0.0s finished
[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done   2 out of  14 | elapsed:    0.0s remaining:    0.1s
[Parallel(n_jobs=14)]: Done  14 out of  14 | elapsed:    0.0s finished
[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parall

ECOD completado en 63.3s
Ofertas con SCORE_ECOD: 3,021,336 (100.0%)
ECOD — anomalías al P5: 151,067 (5.00%)


[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done   2 out of  14 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=14)]: Done  14 out of  14 | elapsed:    0.0s finished
[Parallel(n_jobs=14)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done   2 out of  14 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=14)]: Done  14 out of  14 | elapsed:    0.0s finished


## 6. LOF sobre subconjunto

**LOF (Local Outlier Factor, Breunig et al. 2000):** compara la densidad local de un punto con la de sus *k* vecinos más cercanos. Es $O(n^2)$ en el caso general, lo que lo vuelve inviable sobre 3M de filas, pero perfectamente aplicable sobre el subconjunto formado por las anomalías robustas IF (108 k) más una muestra normal estratificada (50 k).

In [6]:
robustas = pd.read_csv(ROBUSTAS_PATH)
robustas_keys = set(zip(robustas['NRO_SICOP'], robustas['NRO_OFERTA'], robustas['NRO_LINEA']))
df['IS_ROBUST_ANOMALY'] = df.apply(
    lambda r: 1 if (r['NRO_SICOP'], r['NRO_OFERTA'], r['NRO_LINEA']) in robustas_keys else 0,
    axis=1
).astype(int) if False else df[key_cols].apply(tuple, axis=1).isin(robustas_keys).astype(int)

rng = np.random.default_rng(SEMILLA_PRINCIPAL)
normales_pool = df[df['IS_ROBUST_ANOMALY'] == 0]
n_por_segmento = (
    normales_pool['OFERTA_SEGMENTO']
    .value_counts(normalize=True)
    .mul(50_000)
    .round()
    .astype(int)
    .clip(lower=1)
)
muestras_norm = []
for seg, n in n_por_segmento.items():
    sub = normales_pool[normales_pool['OFERTA_SEGMENTO'] == seg]
    n_take = min(n, len(sub))
    if n_take > 0:
        muestras_norm.append(sub.sample(n=n_take, random_state=SEMILLA_PRINCIPAL))
muestra_normal = pd.concat(muestras_norm)

subset_idx = list(set(df.index[df['IS_ROBUST_ANOMALY'] == 1]).union(set(muestra_normal.index)))
subset_idx = sorted(subset_idx)
df['IS_LOF_SUBSET'] = df.index.isin(subset_idx).astype(int)
print(f'Subconjunto LOF: {len(subset_idx):,} ofertas')
print(f'  Anomalías robustas IF: {int(df.loc[subset_idx, "IS_ROBUST_ANOMALY"].sum()):,}')
print(f'  Muestra normal: {len(muestra_normal):,}')

df['SCORE_LOF'] = np.nan
df['ANOM_LOF'] = 0

t0 = time.time()
for seg in segmentos:
    mask = ((df['OFERTA_SEGMENTO'] == seg) & (df['IS_LOF_SUBSET'] == 1)).values
    n = mask.sum()
    if n < 50:
        continue
    X = df.loc[mask, FEATURES_RELATIVAS].values
    n_neighbors = min(20, max(5, n // 50))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=0.10, n_jobs=-1)
    pred = lof.fit_predict(X)
    scores = -lof.negative_outlier_factor_
    df.loc[mask, 'SCORE_LOF'] = scores

umbral_lof = df.loc[df['IS_LOF_SUBSET']==1, 'SCORE_LOF'].quantile(1 - PERCENTIL)
df.loc[(df['IS_LOF_SUBSET']==1) & (df['SCORE_LOF'] >= umbral_lof), 'ANOM_LOF'] = 1

elapsed = time.time() - t0
n_lof = int(df['ANOM_LOF'].sum())
print(f'\nLOF completado en {elapsed:.0f}s')
print(f'LOF — anomalías al P{int(PERCENTIL*100)} (sobre subconjunto): {n_lof:,}')

Subconjunto LOF: 158,230 ofertas
  Anomalías robustas IF: 108,231
  Muestra normal: 49,999

LOF completado en 26s
LOF — anomalías al P5 (sobre subconjunto): 7,906


## 7. Solapamiento Jaccard entre métodos

Para cada par de métodos se calcula el índice de Jaccard $J(A, B) = |A \cap B| / |A \cup B|$ sobre los conjuntos de anomalías al P5. El LOF se compara solo dentro del subconjunto donde se aplicó (158 k filas), por lo que su comparación con IF/HBOS/ECOD también se restringe a ese subconjunto.

In [7]:
set_if = set(df.index[df['ANOMALY_FLAG'] == 1])
set_hbos = set(df.index[df['ANOM_HBOS'] == 1])
set_ecod = set(df.index[df['ANOM_ECOD'] == 1])
set_lof = set(df.index[df['ANOM_LOF'] == 1])
subset_pop = set(df.index[df['IS_LOF_SUBSET'] == 1])

def jaccard(A, B):
    if not (A or B):
        return float('nan')
    return len(A & B) / len(A | B)

metodos_full = {'IF': set_if, 'HBOS': set_hbos, 'ECOD': set_ecod}
labels_full = list(metodos_full.keys())
j_full = pd.DataFrame(index=labels_full, columns=labels_full, dtype=float)
for a in labels_full:
    for b in labels_full:
        j_full.loc[a, b] = jaccard(metodos_full[a], metodos_full[b])

metodos_subset = {'IF': set_if & subset_pop, 'HBOS': set_hbos & subset_pop,
                  'ECOD': set_ecod & subset_pop, 'LOF': set_lof}
labels_sub = list(metodos_subset.keys())
j_sub = pd.DataFrame(index=labels_sub, columns=labels_sub, dtype=float)
for a in labels_sub:
    for b in labels_sub:
        j_sub.loc[a, b] = jaccard(metodos_subset[a], metodos_subset[b])

print('Jaccard sobre dataset completo (3M filas):')
print(j_full.to_string(float_format=lambda x: f'{x:.3f}'))
print('\nJaccard sobre subconjunto LOF (~158k filas):')
print(j_sub.to_string(float_format=lambda x: f'{x:.3f}'))

Jaccard sobre dataset completo (3M filas):
        IF  HBOS  ECOD
IF   1.000 0.088 0.217
HBOS 0.088 1.000 0.170
ECOD 0.217 0.170 1.000

Jaccard sobre subconjunto LOF (~158k filas):
        IF  HBOS  ECOD   LOF
IF   1.000 0.172 0.397 0.061
HBOS 0.172 1.000 0.286 0.081
ECOD 0.397 0.286 1.000 0.086
LOF  0.061 0.081 0.086 1.000


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(j_full.astype(float), annot=True, fmt='.3f', cmap='YlGnBu',
            cbar_kws={'label': 'Jaccard'}, ax=axes[0], vmin=0, vmax=1, square=True)
axes[0].set_title(f'Solapamiento entre métodos al P{int(PERCENTIL*100)} — Dataset completo (3M filas)', fontsize=12)

sns.heatmap(j_sub.astype(float), annot=True, fmt='.3f', cmap='YlGnBu',
            cbar_kws={'label': 'Jaccard'}, ax=axes[1], vmin=0, vmax=1, square=True)
axes[1].set_title(f'Solapamiento — Subconjunto LOF ({len(subset_pop):,} filas)', fontsize=12)

plt.tight_layout()
plt.savefig(FIG_DIR / 'jaccard_metodos_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Guardado: {FIG_DIR / "jaccard_metodos_heatmap.png"}')

Guardado: figuras\jaccard_metodos_heatmap.png


## 8. Diagrama de Venn (IF, HBOS, ECOD)

In [9]:
if HAS_VENN:
    fig, ax = plt.subplots(figsize=(10, 8))
    venn3([set_if, set_hbos, set_ecod], set_labels=('IF', 'HBOS', 'ECOD'), ax=ax)
    ax.set_title(f'Anomalías detectadas al P{int(PERCENTIL*100)} — Tres métodos sobre dataset completo', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'venn_metodos.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Guardado: {FIG_DIR / "venn_metodos.png"}')

interseccion3 = set_if & set_hbos & set_ecod
print(f'\nConsenso multi-método (IF ∩ HBOS ∩ ECOD): {len(interseccion3):,} ofertas ({len(interseccion3)/len(df)*100:.2f}%)')
print(f'  Solo en IF:                {len(set_if - set_hbos - set_ecod):,}')
print(f'  Solo en HBOS:              {len(set_hbos - set_if - set_ecod):,}')
print(f'  Solo en ECOD:              {len(set_ecod - set_if - set_hbos):,}')
print(f'  IF ∩ HBOS (sin ECOD):      {len((set_if & set_hbos) - set_ecod):,}')
print(f'  IF ∩ ECOD (sin HBOS):      {len((set_if & set_ecod) - set_hbos):,}')
print(f'  HBOS ∩ ECOD (sin IF):      {len((set_hbos & set_ecod) - set_if):,}')

Guardado: figuras\venn_metodos.png

Consenso multi-método (IF ∩ HBOS ∩ ECOD): 17,526 ofertas (0.58%)
  Solo en IF:                90,172
  Solo en HBOS:              100,319
  Solo en ECOD:              70,800
  IF ∩ HBOS (sin ECOD):      6,926
  IF ∩ ECOD (sin HBOS):      36,443
  HBOS ∩ ECOD (sin IF):      26,298


## 9. Exportar consenso multi-método

In [10]:
df['IN_INT3'] = df.index.isin(interseccion3).astype(int)
df['IN_LOF'] = df['ANOM_LOF']
df['N_METODOS_FLAG'] = (df['ANOMALY_FLAG'] + df['ANOM_HBOS'] + df['ANOM_ECOD']).astype(int)

consenso_cols = key_cols + ['OFERTA_SEGMENTO', 'OFERTA_CLASE', 'NOMBRE_INSTITUCION', 'NOMBRE_PROVEEDOR',
                            'OFERTA_PRECIO_UNITARIO_CRC', 'RATIO_OFERTADO_VS_ESTIMADO', 'N_OFERENTES_ITEM',
                            'SINGLE_BID_FLAG', 'IS_ROBUST_ANOMALY',
                            'SCORE_MIN', 'SCORE_HBOS', 'SCORE_ECOD', 'SCORE_LOF',
                            'ANOMALY_FLAG', 'ANOM_HBOS', 'ANOM_ECOD', 'ANOM_LOF',
                            'N_METODOS_FLAG', 'IN_INT3']
consenso_cols = [c for c in consenso_cols if c in df.columns]
consenso = df[df['N_METODOS_FLAG'] >= 2][consenso_cols].copy()
consenso.to_csv(RES_DIR / 'consenso_multimetodo.csv', index=False)
print(f'Guardado: {RES_DIR / "consenso_multimetodo.csv"}  ({len(consenso):,} filas con ≥2 métodos)')

interseccion = df[df['IN_INT3'] == 1][consenso_cols].copy()
interseccion.to_csv(RES_DIR / 'interseccion_3metodos.csv', index=False)
print(f'Guardado: {RES_DIR / "interseccion_3metodos.csv"}  ({len(interseccion):,} filas en IF∩HBOS∩ECOD)')

Guardado: resultados\consenso_multimetodo.csv  (87,193 filas con ≥2 métodos)
Guardado: resultados\interseccion_3metodos.csv  (17,526 filas en IF∩HBOS∩ECOD)


## 10. Red flags clásicos vs Isolation Forest

Tres reglas heurísticas inspiradas en la literatura OCDE / Banco Mundial sobre detección de irregularidades en contratación pública. La pregunta es: ¿el IF agrega información sobre estas reglas simples, o es redundante con ellas?

- **RF-1 (Single-bid):** una sola oferta en el ítem.
- **RF-2 (Sobreprecio):** ratio ofertado/estimado > 1.5.
- **RF-3 (Última hora sin competencia):** presentación con menos de 12 horas y sin competencia.

In [11]:
df['RF1_SINGLE_BID'] = (df['SINGLE_BID_FLAG'] == 1).astype(int)
df['RF2_SOBREPRECIO'] = (df['RATIO_OFERTADO_VS_ESTIMADO'] > 1.5).astype(int)
df['RF3_LASTMIN_SOLO'] = ((df['DIAS_PARA_CIERRE'] < 0.5) & (df['SINGLE_BID_FLAG'] == 1)).astype(int)
df['RF_CUALQUIERA'] = ((df['RF1_SINGLE_BID'] + df['RF2_SOBREPRECIO'] + df['RF3_LASTMIN_SOLO']) > 0).astype(int)

rf_summary = []
for rf, name in [('RF1_SINGLE_BID', 'RF-1 Single-bid'),
                  ('RF2_SOBREPRECIO', 'RF-2 Sobreprecio (ratio>1.5)'),
                  ('RF3_LASTMIN_SOLO', 'RF-3 Última hora sin competencia'),
                  ('RF_CUALQUIERA', 'Cualquier red flag')]:
    set_rf = set(df.index[df[rf] == 1])
    n_rf = len(set_rf)
    overlap_if = len(set_rf & set_if)
    j = jaccard(set_rf, set_if)
    rf_summary.append({
        'red_flag': name,
        'n_total': n_rf,
        'pct_dataset': n_rf / len(df) * 100,
        'overlap_con_IF': overlap_if,
        'pct_de_anomalias_IF': overlap_if / max(n_if, 1) * 100,
        'jaccard_con_IF': j,
    })
rf_df = pd.DataFrame(rf_summary)
print('Red flags clásicas y solapamiento con IF:')
print(rf_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
rf_df.to_csv(RES_DIR / 'redflags_vs_if_metricas.csv', index=False)

Red flags clásicas y solapamiento con IF:
                        red_flag  n_total  pct_dataset  overlap_con_IF  pct_de_anomalias_IF  jaccard_con_IF
                 RF-1 Single-bid   383793       12.703           33490               22.169           0.067
    RF-2 Sobreprecio (ratio>1.5)   458356       15.171           59220               39.201           0.108
RF-3 Última hora sin competencia   161904        5.359           13307                8.809           0.044
              Cualquier red flag   794046       26.281           81735               54.105           0.095


In [12]:
set_rf_cualquiera = set(df.index[df['RF_CUALQUIERA'] == 1])

anom_solo_if = set_if - set_rf_cualquiera
anom_if_y_rf = set_if & set_rf_cualquiera
rf_no_if = set_rf_cualquiera - set_if

print(f'Anomalías IF totales: {len(set_if):,}')
print(f'  Capturadas también por al menos un red flag: {len(anom_if_y_rf):,} ({len(anom_if_y_rf)/max(len(set_if),1)*100:.1f}%)')
print(f'  NO capturadas por ningún red flag: {len(anom_solo_if):,} ({len(anom_solo_if)/max(len(set_if),1)*100:.1f}%)')
print(f'\nOfertas con red flag pero NO anómalas en IF: {len(rf_no_if):,}')
print(f'  ({len(rf_no_if)/max(len(set_rf_cualquiera),1)*100:.1f}% del total de red-flagged)')

Anomalías IF totales: 151,067
  Capturadas también por al menos un red flag: 81,735 (54.1%)
  NO capturadas por ningún red flag: 69,332 (45.9%)

Ofertas con red flag pero NO anómalas en IF: 712,311
  (89.7% del total de red-flagged)


In [13]:
if HAS_VENN:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    venn2([set_if, set_rf_cualquiera], set_labels=('IF (P5)', 'Cualquier red flag'), ax=axes[0])
    axes[0].set_title('IF vs reglas heurísticas (cualquiera)', fontsize=12)
    axes[1].axis('off')
    table_data = rf_df[['red_flag', 'n_total', 'pct_dataset', 'pct_de_anomalias_IF', 'jaccard_con_IF']].copy()
    table_data['n_total'] = table_data['n_total'].apply(lambda x: f'{x:,}')
    table_data['pct_dataset'] = table_data['pct_dataset'].apply(lambda x: f'{x:.1f}%')
    table_data['pct_de_anomalias_IF'] = table_data['pct_de_anomalias_IF'].apply(lambda x: f'{x:.1f}%')
    table_data['jaccard_con_IF'] = table_data['jaccard_con_IF'].apply(lambda x: f'{x:.3f}')
    table = axes[1].table(cellText=table_data.values, colLabels=table_data.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.8)
    axes[1].set_title('Resumen cuantitativo de red flags', fontsize=12)
    fig.suptitle('Red flags heurísticos como baseline contra el Isolation Forest', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'redflags_vs_if.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Guardado: {FIG_DIR / "redflags_vs_if.png"}')

Guardado: figuras\redflags_vs_if.png


## 11. Comparación de drivers: HBOS vs SHAP-IF

HBOS no tiene SHAP de árbol, pero sí entrega un *feature score* por feature (la suma de log-inverso de frecuencias antes del agregado). Comparamos qué features impulsan los scores extremos en HBOS vs lo que vimos en IF (vía SHAP). La comparación es orientativa porque las dos cantidades no son directamente equivalentes en magnitud, pero sí en *ranking*.

In [14]:
shap_imp_path = RES_DIR / 'shap_importancia_segmento.csv'
if shap_imp_path.exists():
    shap_imp = pd.read_csv(shap_imp_path)
    rank_shap = shap_imp.sort_values('importancia_seg', ascending=False)
    rank_shap['rank_shap'] = range(1, len(rank_shap)+1)
    print('Comparación informativa: el ranking SHAP-IF se cargó desde el notebook de SHAP.')
    print('Top 15 features por importancia SHAP-IF (Nivel 1):')
    print(rank_shap.head(15)[['feature', 'importancia_seg', 'categoria', 'rank_shap']].to_string(index=False))
else:
    print('No se encontró shap_importancia_segmento.csv — ejecutar primero IF_SICOP_shap.ipynb.')
    rank_shap = None

anom_hbos_subset = df.index[df['ANOM_HBOS'] == 1]
X_hbos_anom = df.loc[anom_hbos_subset, FEATURES_RELATIVAS]
feat_means_hbos = X_hbos_anom.mean()
feat_means_norm = df.loc[df['ANOM_HBOS']==0, FEATURES_RELATIVAS].mean()
feat_diff = (feat_means_hbos - feat_means_norm).abs() / (feat_means_norm.abs() + 1e-9)
feat_diff_df = pd.DataFrame({'feature': FEATURES_RELATIVAS,
                              'diff_relativa': feat_diff.values}).sort_values('diff_relativa', ascending=False)
feat_diff_df['rank_hbos_proxy'] = range(1, len(feat_diff_df)+1)
print('\nTop 15 features por separación HBOS-anom vs HBOS-norm (proxy de importancia):')
print(feat_diff_df.head(15).to_string(index=False, float_format=lambda x: f'{x:.3f}'))
feat_diff_df.to_csv(RES_DIR / 'hbos_features_proxy.csv', index=False)

if rank_shap is not None:
    cmp_drivers = rank_shap[['feature', 'rank_shap']].merge(
        feat_diff_df[['feature', 'rank_hbos_proxy']], on='feature', how='inner'
    )
    cmp_drivers['delta_rank'] = cmp_drivers['rank_shap'] - cmp_drivers['rank_hbos_proxy']
    print('\nComparación de rankings (SHAP-IF vs HBOS-proxy), top 15 SHAP:')
    print(cmp_drivers.sort_values('rank_shap').head(15).to_string(index=False))

Comparación informativa: el ranking SHAP-IF se cargó desde el notebook de SHAP.
Top 15 features por importancia SHAP-IF (Nivel 1):
                    feature  importancia_seg     categoria  rank_shap
 RATIO_OFERTADO_VS_ESTIMADO         0.582094        precio          1
       DIAS_VENTANA_PROCESO         0.519514  temporalidad          2
      LOG_RATIO_VS_ESTIMADO         0.518544        precio          3
            SINGLE_BID_FLAG         0.355218   competencia          4
           DIAS_PARA_CIERRE         0.321570  temporalidad          5
             N_OFERTAS_ITEM         0.289688   competencia          6
                 OFERTA_IVA         0.285305        precio          7
             CV_PRECIO_ITEM         0.278501   competencia          8
           N_OFERENTES_ITEM         0.250921   competencia          9
            RANK_PRECIO_ASC         0.232820   competencia         10
RATIO_OFERTADO_VS_PROM_ITEM         0.231907        precio         11
 N_OFERENTES_ELEGIBLES_ITEM  

## 12. Resumen ejecutivo del experimento

In [17]:
print('=' * 72)
print('RESUMEN EJECUTIVO — Experimento 03 (Benchmark + Red flags)')
print('=' * 72)
print(f'Anomalías al P{int(PERCENTIL*100)}:')
print(f'  Isolation Forest:           {len(set_if):>10,}')
print(f'  HBOS:                       {len(set_hbos):>10,}')
print(f'  ECOD:                       {len(set_ecod):>10,}')
print(f'  LOF (subconjunto):          {len(set_lof):>10,}')
print()
print(f'Consenso multi-método (IF ∩ HBOS ∩ ECOD): {len(interseccion3):,} ({len(interseccion3)/len(df)*100:.2f}%)')
print()
print('Jaccard sobre dataset completo:')
print(j_full.to_string(float_format=lambda x: f'{x:.3f}'))
print()
print('Aporte del IF sobre red flags:')
print(f'  Anomalías IF capturadas también por algún red flag: {len(anom_if_y_rf):,} ({len(anom_if_y_rf)/max(len(set_if),1)*100:.1f}%)')
print(f'  Anomalías IF que NO caen en ningún red flag:        {len(anom_solo_if):,} ({len(anom_solo_if)/max(len(set_if),1)*100:.1f}%)')
print()
print('Artefactos:')
for p in sorted(FIG_DIR.glob('jaccard*')) + sorted(FIG_DIR.glob('venn*')) + sorted(FIG_DIR.glob('redflags*')):
    print(f'  {p}')
for p in sorted(RES_DIR.glob('consenso*')) + sorted(RES_DIR.glob('intersec*')) + sorted(RES_DIR.glob('redflags*')) + sorted(RES_DIR.glob('hbos*')):
    print(f'  {p}')

RESUMEN EJECUTIVO — Experimento 03 (Benchmark + Red flags)
Anomalías al P5:
  Isolation Forest:              151,067
  HBOS:                          151,069
  ECOD:                          151,067
  LOF (subconjunto):               7,906

Consenso multi-método (IF ∩ HBOS ∩ ECOD): 17,526 (0.58%)

Jaccard sobre dataset completo:
        IF  HBOS  ECOD
IF   1.000 0.088 0.217
HBOS 0.088 1.000 0.170
ECOD 0.217 0.170 1.000

Aporte del IF sobre red flags:
  Anomalías IF capturadas también por algún red flag: 81,735 (54.1%)
  Anomalías IF que NO caen en ningún red flag:        69,332 (45.9%)

Artefactos:
  figuras\jaccard_metodos_heatmap.png
  figuras\venn_metodos.png
  figuras\redflags_vs_if.png
  resultados\consenso_multimetodo.csv
  resultados\interseccion_3metodos.csv
  resultados\redflags_vs_if_metricas.csv
  resultados\hbos_features_proxy.csv
